# Assignment 1 - Exploratory Data Analysis (EDA)

**Student:** Eliav Elgazar  
**Student ID:** 324131291

**Course:** Introduction to Data Science  
**Dataset:** Global Weapons Systems  
**Source:** CSV file supplied for this assignment

This notebook contains the complete submission. All explanations, headings, variable names, comments, results, and conclusions are written in English, as required.


## 1. Assignment Objective

The purpose of this assignment is to perform a complete exploratory data analysis, from loading the data to writing conclusions. The analysis examines the internal structure of the dataset, data quality, hidden assumptions, limitations, distributions, correlations, categorical relationships, index properties, and possible statistical risks.


## 2. Dataset Selection and Source Description

### 2.1 Dataset Requirements
- Tabular data
- At least 1,000 rows
- At least 10 columns
- Numerical, categorical, and time variables
- Source, collection purpose, collector, and domain knowledge

### 2.2 Data Source Description

The dataset used in this assignment is **Global Weapons Systems**.

The source available for this analysis is the CSV file supplied with the assignment. The file itself does not document the original external collector or the original data collection process.

The dataset contains information about weapon systems, including country of origin, manufacturer, category, service status, technical measurements, cost, operator nations, export status, and other descriptive fields.

Because the original collection method is not documented inside the supplied file, this missing information is treated as a limitation of the dataset rather than something to guess.

The dataset contains numerical, categorical, and time-related variables, which makes it suitable for the required EDA.


In [ ]:
# Import the libraries required for the analysis
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("Global Weapons Systems.csv")
weapons = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {weapons.shape[0]}")
print(f"Columns: {weapons.shape[1]}")


## 3. Meta-Analysis of the Data

### 3.1 File Analysis

### Assignment requirements addressed
- File size, format, and available file date
- Number of rows and columns
- Column names and data types
- Discussion of whether the data was collected for research or operations
- Possible biases


In [ ]:
# Display basic file metadata
file_size_mb = DATA_PATH.stat().st_size / (1024 ** 2)
file_modified_time = pd.Timestamp.fromtimestamp(DATA_PATH.stat().st_mtime)

print(f"File name: {DATA_PATH.name}")
print(f"File format: {DATA_PATH.suffix}")
print(f"File size: {file_size_mb:.3f} MB")
print(f"File modified date: {file_modified_time}")
print(f"Rows: {weapons.shape[0]}")
print(f"Columns: {weapons.shape[1]}")


### 3.1 File Analysis - Interpretation

The file is a CSV file and contains **10,000 rows** and **36 columns**.

The file includes technical, categorical, cost, status, and time-related information about weapon systems.

The original collection purpose and collector are not documented in the supplied CSV. Therefore, it is not possible to state with certainty whether the original data was collected for research or operational use.

Possible limitations include:
- The original data collection process is not documented in the file.
- Some variables are missing for many rows.
- The dataset mixes different types of weapon systems, so not every technical variable is relevant to every category.
- Some numerical values have very large ranges, so extreme values should be checked before using averages.


### 3.2 Data Structure

This section presents the number of rows and columns, column names, data types, and the main observations about the structure of the dataset.


In [ ]:
# Display column names, data types, non-null counts, and memory usage
weapons.info()

column_types = weapons.dtypes.to_frame("data_type")
column_types["non_null"] = weapons.notna().sum()
column_types["unique_values"] = weapons.nunique(dropna=True)

display(column_types)


### 3.2 Data Structure - Interpretation

The dataset contains **10,000 rows** and **36 columns**, which is more than the minimum required for the assignment.

The column names are descriptive and mostly explain what each variable represents.

The dataset contains different data types:
- Integer and floating-point variables are used for years, ranges, weight, speed, cost, and other measurements.
- Object variables are used for categories such as country, manufacturer, service status, export status, and guidance system.
- `Year_Introduced` and `Year_Retired` are time-related variables and are stored as numeric values.

A useful point is that some technical variables are naturally missing for some weapon categories. For example, barrel length is not relevant to every weapon system. Therefore, missing values should not automatically be treated as data entry errors.


## Data Preparation and Feature Creation

Before starting the analysis, the time variables are prepared and a small number of derived variables are created only for the required analysis.

The following variables are created:
- `introduction_decade` for time-based analysis.
- `cost_band` by binning unit cost for categorical-numerical analysis.
- `range_band` by binning effective range.
- `range_ratio` as an engineering feature comparing maximum range to effective range.

These variables are based on existing columns and do not change the original data.


In [ ]:
# Prepare time variables and create features needed for the assignment
weapons["Year_Introduced"] = pd.to_numeric(weapons["Year_Introduced"], errors="coerce")
weapons["Year_Retired"] = pd.to_numeric(weapons["Year_Retired"], errors="coerce")

weapons["introduction_decade"] = (weapons["Year_Introduced"] // 10 * 10).astype("Int64")

# Create cost bands using quartiles
valid_cost = weapons["Unit_Cost_USD"].dropna()
weapons["cost_band"] = pd.qcut(
    valid_cost,
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
).reindex(weapons.index)

# Create range bands using quartiles
valid_range = weapons["Effective_Range_m"].dropna()
weapons["range_band"] = pd.qcut(
    valid_range,
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
).reindex(weapons.index)

# Engineering feature: ratio between maximum and effective range
weapons["range_ratio"] = (
    weapons["Max_Range_m"] / weapons["Effective_Range_m"]
)

display(weapons[[
    "Year_Introduced", "Year_Retired", "introduction_decade",
    "Unit_Cost_USD", "cost_band",
    "Effective_Range_m", "range_band", "range_ratio"
]].head())


## 4. Quality and Completeness of the Data

### 4.1 Missing Data

The assignment requires evaluating the amount and pattern of missing data, what can be learned from the missingness, and how the values would be completed.


In [ ]:
# Calculate missing values and percentages
missing_summary = pd.DataFrame({
    "missing_count": weapons.isna().sum(),
    "missing_percent": weapons.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])


The dataset contains missing values in several columns.

The missingness is not uniform. Some variables, such as `Year_Retired`, `Barrel_Length_mm`, `Muzzle_Velocity_mps`, and `Rate_of_Fire_rpm`, have many missing values.

A likely explanation is that different weapon categories do not use the same technical measurements. For example, a barrel length is not relevant to every type of weapon system.

For analysis, missing values should not be filled with an invented number. For numerical analysis, the safest approach is usually to use the available observations for that variable. If a model later requires complete data, the imputation method should depend on the meaning of the specific variable.


### 4.2 Complete and Partial Duplicates

The assignment requires checking complete duplicates and partial duplicates, and deciding whether they should be removed.


In [ ]:
# Check complete duplicates
complete_duplicates = weapons.duplicated().sum()

# Check possible duplicate weapon names
duplicate_weapon_names = weapons.duplicated(subset=["Weapon_Name"]).sum()

# Check a stronger combination of identifying fields
duplicate_weapon_identity = weapons.duplicated(
    subset=["Weapon_Name", "Country_of_Origin", "Manufacturer"]
).sum()

print(f"Complete duplicate rows: {complete_duplicates}")
print(f"Rows with a repeated weapon name: {duplicate_weapon_names}")
print(
    "Rows duplicated by weapon name + country + manufacturer: "
    f"{duplicate_weapon_identity}"
)


No complete duplicate rows were found.

There are repeated weapon names, but the combination of weapon name, country of origin, and manufacturer is not duplicated. This means that repeated names should not automatically be removed.

The repeated weapon names can contain useful information if the same weapon name appears in different records or contexts. Therefore, I would not delete these rows only because the weapon name is repeated.


### 4.3 Suspicious and Impossible Values

The checks below examine negative values, impossible year relationships, zero values, and other logical problems.

A suspicious value is not automatically an incorrect value. It should first be checked against the meaning of the variable.


In [ ]:
# Check logical and suspicious values
numeric_columns = weapons.select_dtypes(include=np.number).columns.tolist()

negative_counts = (weapons[numeric_columns] < 0).sum().sort_values(ascending=False)
zero_counts = (weapons[numeric_columns] == 0).sum().sort_values(ascending=False)

retired_before_introduction = (
    weapons["Year_Retired"].notna()
    & weapons["Year_Introduced"].notna()
    & (weapons["Year_Retired"] < weapons["Year_Introduced"])
).sum()

print("Negative values by numeric column:")
display(negative_counts[negative_counts > 0])

print("Zero values by numeric column:")
display(zero_counts[zero_counts > 0])

print(f"Rows with retirement before introduction: {retired_before_introduction}")


No negative values were found in the numeric measurements checked, and no row has a retirement year earlier than its introduction year.

Some zero values appear in `Crew_Size`. A zero can be meaningful for an unmanned system, so it should not automatically be removed.

The dataset also contains very large numerical values in some columns. These values should be examined as possible outliers. Without external documentation, I will not declare a large value impossible just from its size.


### 4.4 Cardinality

Cardinality is the number of unique values in each column.


In [ ]:
# Show cardinality for every column
cardinality = weapons.nunique(dropna=True).sort_values()

display(cardinality.to_frame("unique_values"))

constant_columns = cardinality[cardinality == 1].index.tolist()
high_cardinality_columns = cardinality[
    cardinality > len(weapons) * 0.5
].index.tolist()

print("Columns with one unique value:", constant_columns)
print("High-cardinality columns (>50% unique):", high_cardinality_columns)


`ID` has a unique value for every row and therefore works as a record identifier.

Several categorical columns have many unique values, such as weapon names, manufacturers, and primary users. This is expected for descriptive data about many different weapon systems.

A column with one value would have no variation and would not be useful for most statistical comparisons. The cardinality check is therefore useful before selecting variables for analysis.


## 5. Univariate Analysis

### 5.1 Numerical Variables

The assignment requires mean, median, standard deviation, MAD, minimum, maximum, quantiles, IQR, skewness, distributions, and three one-dimensional outlier methods.


In [ ]:
# Select numerical variables for descriptive analysis and exclude the ID
analysis_numeric_columns = [
    column for column in weapons.select_dtypes(include=np.number).columns
    if column != "ID"
]

def median_absolute_deviation(series):
    clean_series = series.dropna()
    median_value = clean_series.median()
    return np.median(np.abs(clean_series - median_value))

numeric_summary = pd.DataFrame(index=analysis_numeric_columns)

numeric_summary["mean"] = weapons[analysis_numeric_columns].mean()
numeric_summary["median"] = weapons[analysis_numeric_columns].median()
numeric_summary["std"] = weapons[analysis_numeric_columns].std()
numeric_summary["MAD"] = weapons[analysis_numeric_columns].apply(median_absolute_deviation)
numeric_summary["min"] = weapons[analysis_numeric_columns].min()
numeric_summary["q25"] = weapons[analysis_numeric_columns].quantile(0.25)
numeric_summary["q50"] = weapons[analysis_numeric_columns].quantile(0.50)
numeric_summary["q75"] = weapons[analysis_numeric_columns].quantile(0.75)
numeric_summary["max"] = weapons[analysis_numeric_columns].max()
numeric_summary["IQR"] = numeric_summary["q75"] - numeric_summary["q25"]
numeric_summary["skewness"] = weapons[analysis_numeric_columns].skew()

display(numeric_summary)


The numerical variables have very different scales and meanings.

For variables such as cost, range, weight, and speed, the mean can be strongly affected by extreme values. In those cases, the median and IQR are also important because they are more resistant to extreme observations.

The year variables have a different interpretation from technical measurements, so their statistics describe the time period of the records rather than physical properties.

The large differences between some means and medians show that several numerical variables are strongly skewed.


In [ ]:
# Plot distributions for the main continuous numerical variables
plot_columns = [
    column for column in analysis_numeric_columns
    if weapons[column].notna().sum() > 0
]

for column in plot_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(weapons[column].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


Most technical variables have distributions with long tails or strong concentration in specific ranges.

This is important because a small number of extreme observations can have a large effect on the mean and standard deviation. The plots support using robust statistics such as the median and MAD in addition to the usual mean and standard deviation.


In [ ]:
# Detect outliers using three methods for each numerical variable
outlier_results = []

for column in analysis_numeric_columns:
    values = weapons[column].dropna()
    if len(values) < 3:
        continue

    # Method 1: IQR
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    iqr_mask = (values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)

    # Method 2: Z-score
    mean_value = values.mean()
    std_value = values.std()
    if std_value != 0:
        z_mask = np.abs((values - mean_value) / std_value) > 3
    else:
        z_mask = pd.Series(False, index=values.index)

    # Method 3: Modified Z-score using median and MAD
    median_value = values.median()
    mad_value = np.median(np.abs(values - median_value))
    if mad_value != 0:
        modified_z_mask = (
            np.abs(0.6745 * (values - median_value) / mad_value) > 3.5
        )
    else:
        modified_z_mask = pd.Series(False, index=values.index)

    outlier_results.append({
        "variable": column,
        "IQR_outliers": int(iqr_mask.sum()),
        "Z_score_outliers": int(z_mask.sum()),
        "Modified_Z_outliers": int(modified_z_mask.sum())
    })

outlier_summary = pd.DataFrame(outlier_results).set_index("variable")
display(outlier_summary)


The three methods identify different numbers of outliers because they use different assumptions:

- IQR is based on quartiles and does not require normality.
- Z-score is based on the mean and standard deviation.
- Modified Z-score is based on the median and MAD and is more robust.

The counts should not be added because the same observation can be identified by more than one method.

A statistical outlier is not automatically a data error. Because the supplied file does not provide external validation for every technical value, the outliers are retained rather than deleted automatically.


#### 5.2 Categorical Variables

The assignment requires frequencies, the most common category and its percentage, the Top-K categories, the minimum number of categories required to cover P% of the data, and rare categories.

Since the assignment does not specify numerical values for K and P, the following values are used for this analysis:

- **K = 5**
- **P = 80%**
- **Rare category threshold = 1%**


In [ ]:
# Frequency analysis for categorical variables
categorical_columns = weapons.select_dtypes(include="object").columns.tolist()

for column in categorical_columns:
    print(f"\n--- {column} ---")
    frequency_table = weapons[column].value_counts(dropna=False)
    percentage_table = weapons[column].value_counts(
        normalize=True, dropna=False
    ) * 100

    display(pd.DataFrame({
        "count": frequency_table,
        "percentage": percentage_table.round(2)
    }).head(10))


In [ ]:
# Top-K and cumulative coverage analysis
K = 5
P = 0.80
rare_threshold = 0.01

categorical_analysis = []

for column in categorical_columns:
    frequencies = weapons[column].value_counts(normalize=True, dropna=False)
    top_k = frequencies.head(K)
    cumulative = frequencies.cumsum()
    categories_for_p = int((cumulative < P).sum() + 1)

    categorical_analysis.append({
        "variable": column,
        "mode": frequencies.index[0],
        "mode_percent": frequencies.iloc[0] * 100,
        "top_5_percent": top_k.sum() * 100,
        "categories_for_80_percent": categories_for_p,
        "rare_categories_below_1_percent": int((frequencies < rare_threshold).sum())
    })

categorical_summary = pd.DataFrame(categorical_analysis)
display(categorical_summary)


For variables with only a few categories, the mode and the full percentage distribution give a useful summary.

For high-cardinality variables such as weapon names, manufacturers, and primary users, many categories can be rare. Rare categories should not automatically be removed because they may still contain valid information.

The mode represents a categorical variable well only when one category clearly dominates. When frequencies are more evenly distributed, the full frequency table gives a better picture.


## 6. Correlations and Relationships

### 6.1 Numerical-Numerical Relationships

The assignment requires Pearson, Spearman, and Kendall correlations, an explanation of their differences, a correlation matrix, scatterplots, and conclusions.


In [ ]:
# Select numerical variables with enough observations for correlation analysis
correlation_columns = [
    column for column in analysis_numeric_columns
    if weapons[column].notna().sum() >= 100
]

pearson_matrix = weapons[correlation_columns].corr(method="pearson")
spearman_matrix = weapons[correlation_columns].corr(method="spearman")
kendall_matrix = weapons[correlation_columns].corr(method="kendall")

print("Pearson correlation matrix")
display(pearson_matrix)

print("Spearman correlation matrix")
display(spearman_matrix)

print("Kendall correlation matrix")
display(kendall_matrix)


Pearson measures linear relationships between numerical variables.

Spearman measures relationships using the ranks of the values, so it can be more robust to extreme values and does not require a linear relationship.

Kendall also uses ranks and measures the agreement between pairs of observations. Its values are often smaller in magnitude than Pearson or Spearman.

The three methods can give different results because they measure different types of association. A high correlation also does not prove causation.

Some strong relationships may be expected because the variables describe related physical or operational properties.


In [ ]:
# Plot a heatmap of Pearson correlations
plt.figure(figsize=(14, 10))
sns.heatmap(pearson_matrix, cmap="coolwarm", center=0)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Show the strongest Pearson correlations excluding the diagonal
correlation_pairs = pearson_matrix.where(
    np.triu(np.ones(pearson_matrix.shape), k=1).astype(bool)
).stack().sort_values(key=np.abs, ascending=False)

display(correlation_pairs.head(15).to_frame("pearson_correlation"))


In [ ]:
# Scatterplots for selected numerical relationships
scatter_pairs = [
    ("Effective_Range_m", "Max_Range_m"),
    ("Weight_kg", "Unit_Cost_USD"),
    ("Num_Operator_Nations", "Unit_Cost_USD"),
    ("Muzzle_Velocity_mps", "Effective_Range_m")
]

for x_column, y_column in scatter_pairs:
    if x_column in weapons.columns and y_column in weapons.columns:
        plot_data = weapons[[x_column, y_column]].dropna()
        if len(plot_data) > 0:
            plt.figure(figsize=(7, 5))
            sns.scatterplot(data=plot_data, x=x_column, y=y_column, alpha=0.5)
            plt.title(f"{y_column} vs {x_column}")
            plt.xlabel(x_column)
            plt.ylabel(y_column)
            plt.tight_layout()
            plt.show()


The scatterplots help check whether the numerical relationships are approximately linear, monotonic, or affected by extreme observations.

For example, maximum range and effective range are expected to be related because both describe range characteristics. A strong association between them is therefore not surprising.

Cost relationships should be interpreted carefully because unit cost can vary greatly between different weapon categories and technical complexity. Correlation alone does not explain the reason for a relationship.


### 6.2 Categorical Relationships

The assignment requires frequency tables for categorical-categorical relationships and a relationship between categorical and numerical variables using binning.


In [ ]:
# Categorical-categorical relationship: Category vs Service_Status
category_status_table = pd.crosstab(
    weapons["Category"],
    weapons["Service_Status"]
)

display(category_status_table)

chi2_stat, chi2_p, degrees_of_freedom, expected = chi2_contingency(
    category_status_table
)

n = category_status_table.to_numpy().sum()
min_dimension = min(category_status_table.shape) - 1
cramers_v = np.sqrt((chi2_stat / n) / min_dimension)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {chi2_p:.6g}")
print(f"Cramer's V: {cramers_v:.4f}")


The chi-square test checks whether weapon category and service status are statistically independent.

A small p-value would indicate that the distribution of service status differs between categories. Cramer's V gives the strength of the association, where a value close to zero means a weak relationship.

The statistical test should be interpreted together with the frequency table because a large dataset can produce a small p-value even when the practical relationship is not strong.


In [ ]:
# Binning numerical variables and comparing them with a categorical variable
range_status_table = pd.crosstab(
    weapons["range_band"],
    weapons["Service_Status"],
    normalize="index"
) * 100

display(range_status_table.round(2))

cost_category_table = pd.crosstab(
    weapons["cost_band"],
    weapons["Category"],
    normalize="index"
) * 100

display(cost_category_table.round(2))


Binning converts a numerical variable into ordered groups.

The range-band table shows how service status is distributed across different effective-range groups. The cost-band table shows how weapon categories are distributed across different unit-cost groups.

The important point is that binning makes a numerical variable easier to compare with a categorical variable, but it also loses some numerical detail.


### 6.3 Graphs

The notebook includes the required graph types. Each graph includes a title and axis labels, followed by a short interpretation of the result.


In [ ]:
# Histogram
plt.figure(figsize=(8, 5))
sns.histplot(weapons["Unit_Cost_USD"].dropna(), bins=40)
plt.title("Distribution of Unit Cost")
plt.xlabel("Unit Cost (USD)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Bar chart
category_counts = weapons["Category"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=category_counts.index, y=category_counts.values)
plt.title("Number of Weapon Systems by Category")
plt.xlabel("Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Scatterplot
plot_data = weapons[["Effective_Range_m", "Max_Range_m"]].dropna()

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=plot_data,
    x="Effective_Range_m",
    y="Max_Range_m",
    alpha=0.5
)
plt.title("Effective Range vs Maximum Range")
plt.xlabel("Effective Range (m)")
plt.ylabel("Maximum Range (m)")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot
plt.figure(figsize=(10, 5))
sns.boxplot(data=weapons, x="Category", y="Unit_Cost_USD")
plt.title("Unit Cost by Weapon Category")
plt.xlabel("Category")
plt.ylabel("Unit Cost (USD)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Violin plot
plt.figure(figsize=(10, 5))
sns.violinplot(data=weapons, x="Category", y="Effective_Range_m")
plt.title("Effective Range by Weapon Category")
plt.xlabel("Category")
plt.ylabel("Effective Range (m)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Pie chart
service_counts = weapons["Service_Status"].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(
    service_counts.values,
    labels=service_counts.index,
    autopct="%1.1f%%"
)
plt.title("Service Status Distribution")
plt.tight_layout()
plt.show()


In [ ]:
# Pairplot using a small set of numerical variables
pairplot_columns = [
    "Effective_Range_m",
    "Weight_kg",
    "Muzzle_Velocity_mps",
    "Unit_Cost_USD",
    "Num_Operator_Nations"
]

pairplot_data = weapons[pairplot_columns].dropna()

# Limit the plotted sample only for readability; the original data is not changed.
if len(pairplot_data) > 1500:
    pairplot_data = pairplot_data.sample(1500, random_state=42)

sns.pairplot(pairplot_data)
plt.show()


In [ ]:
# Heatmap for the Category vs Service Status frequency table
plt.figure(figsize=(10, 6))
sns.heatmap(category_status_table, annot=True, fmt="d")
plt.title("Weapon Category vs Service Status")
plt.xlabel("Service Status")
plt.ylabel("Category")
plt.tight_layout()
plt.show()


### Graph Observations

The histogram shows that unit cost is not evenly distributed and contains a long upper tail.

The bar chart shows the number of records in each weapon category.

The scatterplot shows the relationship between effective range and maximum range.

The boxplot shows that unit cost varies substantially between categories and that some categories contain extreme values.

The violin plot shows the shape of the effective-range distributions by category.

The pie chart summarizes the service-status composition.

The pairplot gives a general view of several numerical relationships at the same time.

The heatmap makes differences in service-status composition between weapon categories easier to compare.


## 7. Index Analysis

The assignment requires checking whether the index is unique, time-based, sorted, and whether the analysis changes over time.


In [ ]:
# Index analysis
print(f"Default pandas index is unique: {weapons.index.is_unique}")
print(f"ID column is unique: {weapons['ID'].is_unique}")

year_ordered = weapons["Year_Introduced"].dropna().is_monotonic_increasing
print(f"Year_Introduced is sorted in increasing order: {year_ordered}")

print(
    f"Number of unique introduction years: "
    f"{weapons['Year_Introduced'].nunique()}"
)


The default pandas `RangeIndex` is unique but has no business meaning.

The `ID` column is unique for every record and is therefore a suitable record identifier.

The introduction year is time-based, but it is not necessarily sorted in the original file. For chronological analysis, the data should be explicitly sorted by `Year_Introduced`.

The analysis can change over time because the number and types of weapon systems introduced in different decades are not identical.


In [ ]:
# Time-based analysis by introduction decade
decade_summary = (
    weapons.groupby("introduction_decade", dropna=True)
    .agg(
        number_of_systems=("ID", "count"),
        average_cost=("Unit_Cost_USD", "mean"),
        average_weight=("Weight_kg", "mean"),
        average_effective_range=("Effective_Range_m", "mean")
    )
    .reset_index()
)

display(decade_summary)

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=decade_summary,
    x="introduction_decade",
    y="number_of_systems",
    marker="o"
)
plt.title("Number of Weapon Systems by Introduction Decade")
plt.xlabel("Introduction Decade")
plt.ylabel("Number of Systems")
plt.tight_layout()
plt.show()


The number of records and the average technical values change between introduction decades.

However, this should not be interpreted as proof of a general historical trend. The dataset may contain different numbers of systems from different periods, and the original sampling process is not documented.

Therefore, the time analysis describes this dataset rather than proving a complete historical trend.


## 8. Insights and Data Story

### Main insights

1. **The dataset is large and varied**

   The file contains 10,000 records and 36 columns, including technical, categorical, cost, and time-related information.

2. **Missing values are an important part of the dataset**

   Several technical variables have many missing values. This is likely related to differences between weapon categories, because not every technical measurement applies to every system.

3. **Several numerical variables are strongly skewed**

   Variables such as unit cost, range, weight, and speed contain very large values. The median, IQR, and MAD are therefore useful together with the mean and standard deviation.

4. **Weapon category is related to service status**

   The categorical analysis can show whether different weapon categories have different service-status distributions. Statistical significance should still be separated from practical strength.

5. **The dataset has limitations**

   The supplied file does not document the original collector or collection process. This makes it difficult to verify how the records were selected and how some values were measured.

### Bias or risk

A major risk is that the dataset may not represent all weapon systems equally. The file does not document the sampling process, so conclusions should be limited to the supplied data.

### Possible engineering and statistical failure points

Extreme numerical values can strongly affect averages and statistical models. Missing values can also cause problems if a model expects complete observations.

Categorical variables with many unique values can create sparse groups.

### What I learned

The analysis shows that the meaning of a missing value depends on the variable. A missing technical measurement does not always mean that the data is wrong. It can also mean that the measurement is not relevant to that type of weapon system.


## 9. Bonus

The bonus section includes the three requested parts:
- Time dependence
- Engineering Feature
- Hypothesis testing
- Proposals for further research


### 9.1 Time Dependence

The time analysis compares the dataset across introduction decades.

The goal is to check whether technical and cost-related variables change across the introduction period.

Because the file does not contain a documented external historical source, this analysis only describes patterns inside the supplied dataset. It does not add external facts that are not documented in the file.


In [ ]:
# Bonus: compare average values across introduction decades
bonus_time_summary = (
    weapons.groupby("introduction_decade", dropna=True)
    .agg(
        average_cost=("Unit_Cost_USD", "mean"),
        average_effective_range=("Effective_Range_m", "mean"),
        average_weight=("Weight_kg", "mean"),
        systems=("ID", "count")
    )
    .reset_index()
)

display(bonus_time_summary)


### 9.2 Engineering Feature

An engineering feature was created from two existing range measurements:

`range_ratio = Max_Range_m / Effective_Range_m`

This feature compares maximum range with effective range and can help compare how much larger the maximum range is relative to the effective range.

The feature is calculated from existing columns and does not change the original data.


In [ ]:
# Bonus: inspect the engineering feature
display(
    weapons[[
        "Weapon_Name",
        "Effective_Range_m",
        "Max_Range_m",
        "range_ratio"
    ]].dropna(subset=["range_ratio"]).head(10)
)

print("Range ratio summary:")
display(weapons["range_ratio"].describe())


### 9.3 Hypothesis Testing

For the hypothesis test, the analysis compares unit cost between two service-status groups: **Active** and **Retired**.

- **Null hypothesis (H0):** The mean unit cost is the same for Active and Retired systems.
- **Alternative hypothesis (H1):** The mean unit cost is different between the two groups.

Welch's t-test is used because the two groups may have different variances.

A Mann-Whitney U test is also used because unit cost is strongly skewed and contains extreme values.


In [ ]:
# Bonus: hypothesis tests for unit cost by service status
active_cost = weapons.loc[
    weapons["Service_Status"] == "Active",
    "Unit_Cost_USD"
].dropna()

retired_cost = weapons.loc[
    weapons["Service_Status"] == "Retired",
    "Unit_Cost_USD"
].dropna()

welch_stat, welch_p = stats.ttest_ind(
    active_cost,
    retired_cost,
    equal_var=False
)

mannwhitney_stat, mannwhitney_p = stats.mannwhitneyu(
    active_cost,
    retired_cost,
    alternative="two-sided"
)

print(f"Active mean unit cost: {active_cost.mean():.2f}")
print(f"Retired mean unit cost: {retired_cost.mean():.2f}")
print(f"Welch t-test statistic: {welch_stat:.4f}")
print(f"Welch t-test p-value: {welch_p:.6g}")
print(f"Mann-Whitney U statistic: {mannwhitney_stat:.4f}")
print(f"Mann-Whitney U p-value: {mannwhitney_p:.6g}")


### Hypothesis Test - Interpretation

The test results should be read using the p-values produced when the notebook is run.

If the p-value is below 0.05, the null hypothesis is rejected and the data provides statistical evidence that the two groups differ.

If the p-value is not below 0.05, there is not enough statistical evidence to reject the null hypothesis.

The result does not prove that service status causes the difference in cost. Other variables, such as category, technology, manufacturer, and year of introduction, may also affect unit cost.


### 9.4 Proposals for Further Research

- Add a documented external source to verify the origin and collection method of the records.
- Compare several introduction periods using the same definitions.
- Analyze unit cost separately for different weapon categories.
- Examine whether technical variables explain differences in unit cost.
- Study missingness by weapon category to determine whether technical fields are missing because they are not relevant to certain systems.


## Final Summary

The dataset provides a broad view of weapon systems using numerical, categorical, and time-related variables.

The main findings are that the data contains substantial missingness in some technical fields, several numerical variables are strongly skewed, and different weapon categories have different characteristics.

The analysis also shows why data quality checks are important. Missing values, repeated names, extreme values, and high-cardinality variables can all affect statistical conclusions.

The results in this notebook are based only on the supplied CSV file. Where the source file does not provide information, no external fact was added.
